```mermaid
graph LR
 日本語+Pythonのコーパス作成 --> トークナイザーの学習
 トークナイザーの学習 --> GPTモデルの設計
  GPTモデルの設計 --> 学習ループの実装
  学習ループの実装 --> ファインチューニング
  ファインチューニング --> 推論
```

| Step | Name |Description |
| ---- | ----------- |----------- |
| 1 | 日本語+Pythonのコーパス作成 | (1)日本語(20~200MB) --> Wikipedia, 青空文庫, ニュース系コーパス, (2)Python(20~200MB) --> GitHub, Kaggle Notebooks |
|2 | トークナイザーの学習 | 語彙数(16k~32k), 日本語(BPE < Unigram), Python code(インデントや記号を分離しすぎない) |
|3| GPTモデルの設計 | Embedding(token embedding, position embedding), Transformer Block(Multi-Head Attention, MLP, Layer Normalization, Residual), Language Model Heads(Softmax) |
|4| 学習ループの実装 | batch size(16~48), sequence length(256~512), learning time(1h~1day) |
|5| ファインチューニング | Kaggle Notebooks, Python code, Python Q&A |
|6| 推論 ||

In [ ]:
from pathlib import Path
from typing import Iterable, Iterator

from datasets import load_dataset

# 80% Japanese text / 20% Python code
JAPANESE_DATASETS = [
    {"name": "mini97/filtered_japanese-wikipedia", "columns": ["text", "content", "body"], "preserve_newlines": False},
    {"name": "globis-university/aozorabunko-clean", "columns": ["body", "text", "content"], "preserve_newlines": False},
]

PYTHON_DATASETS = [
    {"name": "Arjun-G-Ravi/Python-codes", "columns": ["code", "text", "content", "body"], "preserve_newlines": True},
]

def iter_text_fields(row: dict, preferred: Iterable[str]) -> Iterator[str]:
    """Yield text-like fields from a row using preferred column order."""
    for key in preferred:
        if key not in row:
            continue
        value = row[key]
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            for v in value:
                if v is not None:
                    yield str(v)
        else:
            yield str(value)

    if not any(key in row for key in preferred):
        for v in row.values():
            if isinstance(v, str):
                yield v
                break

def clean_text(text: str, preserve_newlines: bool) -> str:
    text = text.replace("\r", "")
    if preserve_newlines:
        lines = [line.rstrip() for line in text.splitlines()]
        return "\n".join(lines).strip()
    return " ".join(text.split())

def stream_dataset(cfg: dict, streaming: bool = True) -> Iterator[str]:
    ds = load_dataset(cfg["name"], streaming=streaming)
    parts = ds.items() if isinstance(ds, dict) else [("data", ds)]
    for split_name, split_ds in parts:
        print(f" -> streaming {cfg['name']} [{split_name}]")
        for row in split_ds:
            for text in iter_text_fields(row, cfg["columns"]):
                cleaned = clean_text(text, cfg.get("preserve_newlines", False))
                if cleaned:
                    yield cleaned

def stream_dataset_multi(configs, streaming: bool = True) -> Iterator[str]:
    for cfg in configs:
        yield from stream_dataset(cfg, streaming=streaming)

def take_n(iterator: Iterator[str], n: int) -> list[str]:
    return [x for _, x in zip(range(n), iterator)]

def build_corpus(out_path: str = "corpus.txt", total_items: int = 100_000, streaming: bool = True) -> None:
    """
    Merge datasets into one text corpus with a clear Japanese section and Python section.
    - total_items: desired total samples (approx). 80% Japanese, 20% Python.
    """
    out_path = Path(out_path)
    target_py = max(int(total_items * 0.2), 1)

    py_iter = stream_dataset_multi(PYTHON_DATASETS, streaming=streaming)
    python_samples = take_n(py_iter, target_py)
    if not python_samples:
        raise RuntimeError("Pythonデータが空でした")

    target_jp = len(python_samples) * 4  # 80% vs 20%
    ja_iter = stream_dataset_multi(JAPANESE_DATASETS, streaming=streaming)
    japanese_samples = take_n(ja_iter, target_jp)

    if not japanese_samples:
        raise RuntimeError("日本語データが空でした")
    if len(japanese_samples) < target_jp:
        print("[warn] 日本語データが不足しています。比率が80%未満になります。")

    with out_path.open("w", encoding="utf-8") as f:
        f.write("# ==== 日本語文章（80%） ====\n")
        for line in japanese_samples:
            f.write(f"{line}\n")
        f.write("\n# ==== Pythonコード（20%） ====\n<code>\n")
        for snippet in python_samples:
            f.write(snippet + "\n\n")
        f.write("</code>\n")

    total_written = len(japanese_samples) + len(python_samples)
    print(
        f"書き出し完了: {out_path} (合計 {total_written} サンプル, 日本語 {len(japanese_samples)}, Python {len(python_samples)})"
    )

# 実行例: 比率は80/20で約10万サンプルを抽出
# build_corpus("corpus.txt", total_items=100_000, streaming=True)
